# 01 - Product Space

Build the product-space network for each market.

- Nodes: Category 3 products
- Node color: Category 2 (most common)
- Edge weight: Hidalgo product proximity phi
- RCA input: choose `counts` or `binary` in the config cell
- Outputs: phi matrix, phi edgelist, GraphML, per-market PNG, and a 2x2 grid PNG


In [ ]:
# --- config ---
HIDALGO_RCA_INPUT = "counts"  # "counts" | "binary"
DISPARITY_ALPHA = 0.05
PLOT_GRAPH = "disparity_or_full"  # "full_gcc" | "disparity_or_full"
MIN_DISPARITY_NODES = 15
SEED = 42
TOP_LABELS_PER_MARKET = 6
LABEL_FONT_SIZE = 12

from pathlib import Path
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
from matplotlib import colors as mcolors
from matplotlib.patches import Patch
import matplotlib.patheffects as pe
from adjustText import adjust_text
import networkx as nx
import numpy as np
import pandas as pd

NOTEBOOK_DIR = Path.cwd() / "notebooks" if (Path.cwd() / "notebooks").is_dir() else Path.cwd()
import sys
sys.path.insert(0, str(NOTEBOOK_DIR))

from dnm_viz_utils import (
    repo_root,
    load_master_drugs,
    load_category2_most_common_mapping,
    load_category_color_map,
    MARKETS,
    MARKET_FILE_NAMES,
)

ROOT = repo_root()
OUT = ROOT / "prescription_reports/new_category2_networks"
FIG = OUT / "figures"
TAB = OUT / "tables"
NET = OUT / "networks"
POS = TAB / "layout_positions"
for p in (FIG, TAB, NET, POS):
    p.mkdir(parents=True, exist_ok=True)

SUFFIX = "" if HIDALGO_RCA_INPUT == "counts" else f"_{HIDALGO_RCA_INPUT}"
print("Output:", OUT)
print("Hidalgo RCA input:", HIDALGO_RCA_INPUT)


In [ ]:
def add_listing_id(df: pd.DataFrame) -> pd.Series:
    website = df["Website"].fillna("").astype(str).str.strip()
    pages = df["Pages"].fillna("").astype(str).str.strip()
    vendor = df["ID_Num"].fillna("").astype(str).str.strip()
    names = df["Productname"].fillna("").astype(str).str.strip().str.casefold()
    cat3 = df["Category 3"].fillna("").astype(str).str.strip()
    fallback = website + " | " + vendor + " | " + names + " | " + cat3
    valid_pages = pages.ne("") & ~pages.str.lower().isin(["nan", "none"])
    out = pages.where(valid_pages, fallback)
    return out.where(~website.isin(["WhiteHouse Market", "ToRReZ"]), fallback).astype(str).str.strip()


def vendor_product_listing_counts(df_market: pd.DataFrame) -> pd.DataFrame:
    d = df_market.dropna(subset=["ID_Num", "Category 3"]).copy()
    d["ID_Num"] = d["ID_Num"].astype(str).str.strip()
    d["Category 3"] = d["Category 3"].astype(str).str.strip()
    d["_listing_id"] = add_listing_id(d)
    d = d[(d["ID_Num"] != "") & (d["Category 3"] != "") & (d["_listing_id"] != "")]
    out = d.groupby(["ID_Num", "Category 3"], sort=False)["_listing_id"].nunique().reset_index(name="x_vp")
    return out[out["x_vp"] > 0]


def counts_to_matrix(counts: pd.DataFrame) -> Tuple[np.ndarray, List[str], List[str]]:
    vendors = counts["ID_Num"].astype(str).str.strip().unique().tolist()
    products = counts["Category 3"].astype(str).str.strip().unique().tolist()
    vendor_i = {v: i for i, v in enumerate(vendors)}
    product_j = {p: j for j, p in enumerate(products)}
    X = np.zeros((len(vendors), len(products)), dtype=np.float64)
    for _, row in counts.iterrows():
        X[vendor_i[str(row["ID_Num"]).strip()], product_j[str(row["Category 3"]).strip()]] = float(row["x_vp"])
    return X, vendors, products


def prepare_rca_input(X_counts: np.ndarray) -> np.ndarray:
    if HIDALGO_RCA_INPUT == "counts":
        return X_counts.astype(np.float64)
    if HIDALGO_RCA_INPUT == "binary":
        return (X_counts > 0).astype(np.float64)
    raise ValueError("HIDALGO_RCA_INPUT must be 'counts' or 'binary'")


def rca_binary_matrix(X: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    row_sum = X.sum(axis=1, keepdims=True)
    col_sum = X.sum(axis=0, keepdims=True)
    grand = X.sum()
    with np.errstate(divide="ignore", invalid="ignore"):
        rca = np.where((row_sum > 0) & (col_sum > 0) & (grand > 0), X * grand / (row_sum * col_sum), 0.0)
    return rca, (rca >= 1.0).astype(np.int8)


def phi_proximity(M: np.ndarray) -> np.ndarray:
    col_sums = M.sum(axis=0).astype(np.float64)
    cooccurrence = M.astype(np.float64).T @ M.astype(np.float64)
    with np.errstate(divide="ignore", invalid="ignore"):
        p_i_given_j = np.divide(cooccurrence, col_sums[np.newaxis, :], out=np.zeros_like(cooccurrence), where=col_sums[np.newaxis, :] > 0)
        p_j_given_i = np.divide(cooccurrence, col_sums[:, np.newaxis], out=np.zeros_like(cooccurrence), where=col_sums[:, np.newaxis] > 0)
    phi = np.minimum(p_i_given_j, p_j_given_i)
    np.fill_diagonal(phi, 0.0)
    return phi


def phi_to_graph(products: List[str], phi: np.ndarray, min_phi: float = 0.0) -> nx.Graph:
    G = nx.Graph()
    G.add_nodes_from(products)
    for i, a in enumerate(products):
        for j in range(i + 1, len(products)):
            w = float(phi[i, j])
            if w > min_phi:
                G.add_edge(a, products[j], phi=w, weight=w)
    return G


def giant_component(G: nx.Graph) -> nx.Graph:
    if G.number_of_nodes() == 0:
        return G.copy()
    if nx.is_connected(G):
        return G.copy()
    return G.subgraph(max(nx.connected_components(G), key=len)).copy()


In [ ]:
def disparity_backbone(G: nx.Graph, alpha: float = DISPARITY_ALPHA, max_iter: int = 50) -> nx.Graph:
    H = G.copy()
    for _ in range(max_iter):
        to_remove = []
        for u, v, data in H.edges(data=True):
            w = float(data.get("weight", 0.0))
            if w <= 0:
                to_remove.append((u, v))
                continue
            keep = True
            for node in (u, v):
                degree = H.degree(node)
                if degree <= 1:
                    continue
                strength = sum(float(H[node][nbr].get("weight", 0.0)) for nbr in H[node])
                if strength <= 0:
                    keep = False
                    break
                p = min(max(w / strength, 1e-15), 1.0 - 1e-15)
                if (1.0 - p) ** (degree - 1) >= alpha:
                    keep = False
                    break
            if not keep:
                to_remove.append((u, v))
        if not to_remove:
            break
        H.remove_edges_from(to_remove)
    H.remove_nodes_from(list(nx.isolates(H)))
    return H


def save_hidalgo_outputs(market: str, products: List[str], phi: np.ndarray, G_full: nx.Graph, G_backbone: nx.Graph, G_plot: nx.Graph, prod_to_cat2: dict) -> None:
    tag = MARKET_FILE_NAMES[market]
    pd.DataFrame(phi, index=products, columns=products).to_csv(TAB / f"{tag}_hidalgo{SUFFIX}_phi_matrix.csv")

    edge_rows = [{"product_i": u, "product_j": v, "phi": float(d.get("phi", d.get("weight", 0.0)))} for u, v, d in G_full.edges(data=True)]
    pd.DataFrame(edge_rows).to_csv(TAB / f"{tag}_hidalgo{SUFFIX}_phi_edgelist.csv", index=False)

    backbone_rows = [{"product_i": u, "product_j": v, "phi": float(d.get("phi", d.get("weight", 0.0)))} for u, v, d in G_backbone.edges(data=True)]
    pd.DataFrame(backbone_rows).to_csv(TAB / f"{tag}_hidalgo{SUFFIX}_disparity_backbone_edges.csv", index=False)

    for name, graph in [("full_phi", G_full), ("disparity_backbone", G_backbone), ("plot_graph", G_plot)]:
        H = graph.copy()
        degree_centrality = nx.degree_centrality(H) if H.number_of_nodes() > 1 else {n: 0.0 for n in H}
        for n in H.nodes():
            H.nodes[n].update({
                "label": str(n),
                "category2": prod_to_cat2.get(str(n), "Unknown"),
                "degree": int(H.degree(n)),
                "degree_centrality": float(degree_centrality.get(n, 0.0)),
            })
        nx.write_graphml(H, NET / f"{tag}_hidalgo{SUFFIX}_{name}.graphml")


In [ ]:
def layout_path(market: str) -> Path:
    tag = MARKET_FILE_NAMES[market]
    return POS / f"{tag}_hidalgo{SUFFIX}_plot_spring_seed{SEED}.json"


def get_layout(market: str, G: nx.Graph) -> dict:
    import json
    path = layout_path(market)
    if path.exists():
        raw = json.loads(path.read_text())
        cached = {k: (float(v[0]), float(v[1])) for k, v in raw.items() if isinstance(v, list) and len(v) == 2}
        if set(cached) == set(map(str, G.nodes())):
            return cached
    pos = nx.spring_layout(G, seed=SEED, weight="weight", iterations=300)
    pos = {str(k): (float(v[0]), float(v[1])) for k, v in pos.items()}
    path.write_text(json.dumps({k: [float(v[0]), float(v[1])] for k, v in pos.items()}, indent=2, sort_keys=True))
    return pos


def darker(color, factor=0.72):
    r, g, b = mcolors.to_rgb(color)
    return mcolors.to_hex((r * factor, g * factor, b * factor))



def draw_adjusted_labels(ax, G: nx.Graph, pos: dict, degrees: dict) -> None:
    top_nodes = sorted(G.nodes(), key=lambda n: degrees[n], reverse=True)[:TOP_LABELS_PER_MARKET]
    if not top_nodes:
        return

    texts = []
    x_all = [pos[n][0] for n in G.nodes()]
    y_all = [pos[n][1] for n in G.nodes()]
    target_x = []
    target_y = []

    for i, node in enumerate(top_nodes):
        x, y = pos[node]
        angle = 2 * np.pi * i / max(1, len(top_nodes))
        label_x = x + 0.08 * np.cos(angle)
        label_y = y + 0.08 * np.sin(angle)
        text = ax.text(
            label_x,
            label_y,
            str(node)[:30],
            fontsize=LABEL_FONT_SIZE,
            color="#1f1f1f",
            ha="center",
            va="center",
            zorder=10,
            path_effects=[pe.withStroke(linewidth=3.0, foreground="white")],
        )
        texts.append(text)
        target_x.append(x)
        target_y.append(y)

    adjust_text(
        texts,
        x=x_all,
        y=y_all,
        target_x=target_x,
        target_y=target_y,
        ax=ax,
        expand_points=(2.0, 2.0),
        expand_text=(1.35, 1.55),
        force_points=(0.45, 0.55),
        force_text=(0.7, 0.9),
        lim=600,
        arrowprops=dict(arrowstyle="-", color="#333333", lw=0.45, alpha=0.75, shrinkA=6, shrinkB=4),
    )

def draw_hidalgo_graph(ax, G: nx.Graph, market: str, prod_to_cat2: dict, color_map: dict, title: str) -> None:
    if G.number_of_nodes() == 0:
        ax.axis("off")
        return
    G = giant_component(G)
    pos = get_layout(market, G)
    categories = {n: prod_to_cat2.get(str(n), "Unknown") for n in G.nodes()}
    degrees = dict(G.degree())
    max_degree = max(degrees.values()) if degrees else 1
    weights = [float(d.get("phi", d.get("weight", 0))) for _, _, d in G.edges(data=True)]
    max_weight = max(weights) if weights else 1

    edge_widths = [0.2 + 2.4 * float(G[u][v].get("phi", 0)) / max_weight for u, v in G.edges()]
    nx.draw_networkx_edges(G, pos, ax=ax, edge_color="#c9c9c9", alpha=0.42, width=edge_widths)

    for cat in sorted(set(categories.values()), key=lambda c: (c != "Prescription", c)):
        nodes = [n for n in G.nodes() if categories[n] == cat]
        if not nodes:
            continue
        color = color_map.get(cat, "#B3B3B3")
        sizes = [90 + 840 * np.sqrt(degrees[n] / max_degree) for n in nodes]
        nx.draw_networkx_nodes(
            G,
            pos,
            nodelist=nodes,
            node_size=sizes,
            node_color=[color] * len(nodes),
            edgecolors=[darker(color)] * len(nodes),
            linewidths=0.6,
            alpha=0.96,
            ax=ax,
        )

    draw_adjusted_labels(ax, G, pos, degrees)
    ax.set_title(title, loc="left", fontsize=12)
    ax.axis("off")


In [ ]:
def build_market_hidalgo(market: str, df_drugs: pd.DataFrame, prod_to_cat2: dict):
    df_m = df_drugs[df_drugs["Website"] == market].copy()
    counts = vendor_product_listing_counts(df_m)
    if counts.empty:
        return None

    X_counts, vendors, products = counts_to_matrix(counts)
    X = prepare_rca_input(X_counts)
    rca, M = rca_binary_matrix(X)
    phi = phi_proximity(M)
    G_full = phi_to_graph(products, phi)
    G_full_gcc = giant_component(G_full)
    G_backbone = disparity_backbone(G_full, alpha=DISPARITY_ALPHA)
    G_backbone_gcc = giant_component(G_backbone)

    if PLOT_GRAPH == "full_gcc":
        G_plot = G_full_gcc
    elif PLOT_GRAPH == "disparity_or_full":
        G_plot = G_backbone_gcc if G_backbone_gcc.number_of_nodes() >= MIN_DISPARITY_NODES else G_full_gcc
    else:
        raise ValueError("PLOT_GRAPH must be 'full_gcc' or 'disparity_or_full'")

    save_hidalgo_outputs(market, products, phi, G_full, G_backbone, G_plot, prod_to_cat2)
    return {
        "products": products,
        "vendors": vendors,
        "counts": counts,
        "G_full": G_full,
        "G_backbone": G_backbone,
        "G_plot": G_plot,
    }


df_all = load_master_drugs()
df_all = df_all[df_all["Website"].isin(MARKETS)].copy()
prod_to_cat2 = load_category2_most_common_mapping(df_all)
color_map = load_category_color_map(prod_to_cat2)
print(f"Loaded {len(df_all):,} drug rows and {len(prod_to_cat2)} Category 3 -> Category 2 mappings")

results = {}
for market in MARKETS:
    result = build_market_hidalgo(market, df_all, prod_to_cat2)
    if result is None:
        print(market, "no Hidalgo graph")
        continue
    results[market] = result
    G_plot = result["G_plot"]
    fig, ax = plt.subplots(figsize=(12, 10), facecolor="white")
    draw_hidalgo_graph(ax, G_plot, market, prod_to_cat2, color_map, market)
    out_png = FIG / f"{MARKET_FILE_NAMES[market]}_hidalgo{SUFFIX}_product_space.png"
    fig.savefig(out_png, dpi=300, bbox_inches="tight", facecolor="white")
    plt.show()
    plt.close(fig)
    print(
        f"{market}: full {result['G_full'].number_of_nodes()} nodes/{result['G_full'].number_of_edges()} edges; "
        f"backbone {result['G_backbone'].number_of_nodes()} nodes/{result['G_backbone'].number_of_edges()} edges; "
        f"plot {G_plot.number_of_nodes()} nodes/{G_plot.number_of_edges()} edges"
    )


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(20, 18), facecolor="white")
for ax, market, letter in zip(axes.ravel(), MARKETS, "ABCD"):
    result = results.get(market)
    if result is None:
        ax.axis("off")
        continue
    draw_hidalgo_graph(ax, result["G_plot"], market, prod_to_cat2, color_map, "")
    ax.text(0.01, 0.99, letter, transform=ax.transAxes, ha="left", va="top", fontsize=30, color="black")
    ax.text(0.075, 0.99, market, transform=ax.transAxes, ha="left", va="top", fontsize=19, color="black")

legend_categories = sorted(
    {prod_to_cat2.get(str(n), "Unknown") for r in results.values() for n in r["G_plot"].nodes()},
    key=lambda c: (c != "Prescription", c),
)
handles = [Patch(facecolor=color_map.get(c, "#B3B3B3"), edgecolor="none", label=c) for c in legend_categories]
fig.legend(
    handles=handles,
    title="Category 2",
    loc="lower center",
    bbox_to_anchor=(0.5, 0.01),
    ncol=5,
    frameon=False,
    fontsize=16,
    title_fontsize=17,
)
fig.tight_layout(rect=(0, 0.08, 1, 1))
fig.subplots_adjust(hspace=0.05, wspace=0.05)
grid_path = FIG / f"hidalgo{SUFFIX}_product_space_2x2.png"
fig.savefig(grid_path, dpi=300, bbox_inches="tight", facecolor="white")
plt.show()
plt.close(fig)
print("Saved", grid_path)
